# 🔒 Mini-MSR: Minimum Security Review

A lightweight security review framework for **Power Platform**, **UiPath**, and **Snowflake Python** automations.

## How to Use

1. **Run All Cells** (`Kernel → Restart & Run All`)
2. **Fill the form** tab-by-tab in Sections A–I, OR upload a pre-filled Excel template on the **📂 Import** tab to auto-populate the form
3. Click **📄 Generate Report** to calculate your security tier and render the Mini-MSR report
4. Click **⬇️ Export HTML** to download the report for sign-off and filing

> **Framework references:** NIST CSF 2.0 · NIST SP 800-218 (SSDF) · NIST SP 800-30 · ISO/IEC 27001:2022
> OWASP LCNC Top 10 · OWASP Top 10 for Agentic Applications · Microsoft Power Platform Well-Architected
> UiPath Automation Ops · Snowflake Horizon Data Governance

In [ ]:
%pip install -q pandas openpyxl ipywidgets

import io
import base64
from datetime import date, datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

# ── Global state ──────────────────────────────────────────────────
_WIDGET_REGISTRY: dict = {}   # field_id → widget instance
_LAST_TIER: list = [0]         # list so helpers can mutate across cells
_DIVIDER = widgets.HTML("<hr style='margin:8px 0;border-color:#dee2e6'>")
_BANNER_STYLE = (
    "background:linear-gradient(135deg,#0c3483 0%,#a2b6df 100%);"
    "padding:20px;border-radius:8px;color:white;margin-bottom:16px"
)

In [ ]:
# ── Questionnaire schema ─────────────────────────────────────────
# field types: text | textarea | dropdown | multicheck | yesno | yesno_text | date
QUESTIONNAIRE_SCHEMA = {
    "A": {
        "title": "Automation Identification", "icon": "📋",
        "fields": [
            {"id":"A1","label":"Automation Name & Unique ID","type":"text","required":True,
             "hint":"Use CoE naming convention, e.g. FIN-AP-Invoice-Reconcile-v1","tier_weight":None},
            {"id":"A2","label":"Business Sponsor / Owner","type":"text","required":True,
             "hint":"Accountable executive","tier_weight":None},
            {"id":"A3","label":"Technical Maker / Developer(s)","type":"text","required":True,
             "hint":"Primary + backup contact","tier_weight":None},
            {"id":"A4","label":"Platform(s) Used","type":"multicheck","required":True,"tier_weight":"platform",
             "options":["Power Apps (canvas)","Power Apps (model-driven)","Power Automate (cloud flow)",
                        "Power Automate (desktop)","Copilot Studio","UiPath (attended)","UiPath (unattended)",
                        "Python on Snowflake","Snowpark","Hybrid"]},
            {"id":"A5","label":"Business Purpose","type":"textarea","required":True,
             "placeholder":"Describe the problem this automation solves (≤200 words)","tier_weight":None},
            {"id":"A6","label":"In-Scope Environments","type":"multicheck","tier_weight":None,
             "options":["Dev","Test","Prod"],"hint":"List Tenant IDs in notes field"},
            {"id":"A7","label":"Target Go-Live Date","type":"date","tier_weight":None},
            {"id":"A8","label":"Expected Lifespan","type":"dropdown","required":True,"tier_weight":"lifespan",
             "options":["","Temporary (<6 months)","Ongoing","Mission-critical"]},
            {"id":"A9","label":"Similar / Duplicate Automations Considered?","type":"yesno_text",
             "hint":"Prevents sprawl; mirrors Automation Hub duplicate-check practice","tier_weight":None},
        ]
    },
    "B": {
        "title": "Data Access & Classification", "icon": "🗄️",
        "fields": [
            {"id":"B1","label":"Data Read by This Automation","type":"textarea","required":True,
             "placeholder":"List systems and data types (e.g. Snowflake HR schema, SharePoint list)","tier_weight":None},
            {"id":"B2","label":"Data Written or Modified","type":"textarea",
             "placeholder":"List target systems and data types","tier_weight":None},
            {"id":"B3","label":"Regulated Data Types Processed","type":"multicheck","required":True,"tier_weight":"data_type",
             "options":["PII","PHI","PCI","Financial","Intellectual Property","SOX-relevant","Export-controlled","None"]},
            {"id":"B4","label":"Highest Data Sensitivity Level","type":"dropdown","required":True,"tier_weight":"sensitivity",
             "options":["","Public","Internal","Confidential","Restricted"]},
            {"id":"B5","label":"Volume of Records per Run / per Day","type":"text",
             "placeholder":"e.g. 500/run, 2 000/day","tier_weight":"volume"},
            {"id":"B6","label":"Data Residency Constraints","type":"textarea",
             "placeholder":"Region + regulation, e.g. EU (GDPR), US-only","tier_weight":None},
            {"id":"B7","label":"Retention Period Required","type":"text",
             "placeholder":"e.g. 7 years, 90 days, indefinite","tier_weight":None},
            {"id":"B8","label":"Applicable Regulatory Frameworks","type":"multicheck","required":True,"tier_weight":"compliance",
             "options":["HIPAA","GDPR","CCPA","PCI DSS","SOX","GLBA","FedRAMP","State Privacy Laws","None"]},
            {"id":"B9","label":"Data Storage Destinations","type":"multicheck","tier_weight":None,
             "options":["Dataverse","Snowflake","SharePoint","Azure Blob","External Storage","None"]},
            {"id":"B10","label":"Data Encrypted at Rest and In Transit?","type":"yesno_text","tier_weight":None,
             "hint":"Note key type: Microsoft-managed vs. Customer-Managed Key (CMK)"},
        ]
    },
    "C": {
        "title": "Integrations & API Connections", "icon": "🔌",
        "fields": [
            {"id":"C1","label":"All Connectors / APIs / Data Sources","type":"textarea","required":True,
             "placeholder":"List each; for Power Platform specify standard/premium/custom/HTTP","tier_weight":"integration_type"},
            {"id":"C2","label":"Authentication Method per Connector","type":"textarea","required":True,
             "placeholder":"e.g. Entra ID interactive, service principal, OAuth, API key, managed identity","tier_weight":None},
            {"id":"C3","label":"Any Connections Implicitly Shared (author credentials reused at runtime)?","type":"yesno_text",
             "hint":"Microsoft explicitly discourages this outside narrow public-data scenarios","tier_weight":"implicit_sharing"},
            {"id":"C4","label":"Automation Makes Egress Calls Outside Corporate Tenant?","type":"yesno_text",
             "hint":"List destinations and ports if Yes","tier_weight":"egress"},
            {"id":"C5","label":"Required Connectors Permitted Under Active DLP Policy?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","Unknown / Not Yet Checked"]},
            {"id":"C6","label":"Snowflake: EXTERNAL ACCESS INTEGRATION Objects Required","type":"textarea",
             "placeholder":"N/A if not applicable — list EAI names, network rules, secrets","tier_weight":None},
            {"id":"C7","label":"UiPath: Orchestrator Assets & Credential Stores Used","type":"textarea",
             "placeholder":"N/A if not applicable — list asset names and vault type","tier_weight":None},
        ]
    },
    "D": {
        "title": "Credentials & Secrets", "icon": "🔑",
        "fields": [
            {"id":"D1","label":"Secrets Stored in Code / Config Files?","type":"yesno_text","tier_weight":"secrets_in_code",
             "hint":"Passwords, keys, tokens, certificates — answer Yes if ANY are hardcoded"},
            {"id":"D2","label":"Where Are Secrets Stored?","type":"multicheck","tier_weight":None,
             "options":["Azure Key Vault","AWS Secrets Manager","Google Secret Manager",
                        "Orchestrator Credential Store","Snowflake SECRET object",
                        "HashiCorp Vault","CyberArk","Other / Not Vaulted"]},
            {"id":"D3","label":"Snowflake: Key-Pair or OAuth / WIF Used (not password-only)?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a Snowflake automation"],
             "hint":"Snowflake is deprecating password-only service logins"},
            {"id":"D4","label":"Snowflake: Service Account Set to TYPE = SERVICE?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A"],
             "hint":"Forces key-pair or OAuth and disables password+MFA"},
            {"id":"D5","label":"Key / Certificate Rotation Cadence","type":"dropdown","tier_weight":None,
             "options":["","90 days","180 days","365 days","Not defined","N/A"]},
            {"id":"D6","label":"Service Account Owner + Human Backup","type":"text","required":True,
             "placeholder":"e.g. svc-fin-ap@corp.com — Owner: Jane Smith, Backup: John Doe","tier_weight":None},
        ]
    },
    "E": {
        "title": "Identity & Access", "icon": "🔐",
        "fields": [
            {"id":"E1","label":"Who Can Run This Automation?","type":"textarea","required":True,"tier_weight":"user_scope",
             "placeholder":"Entra groups, Dataverse teams, UiPath folder roles, Snowflake roles"},
            {"id":"E2","label":"Who Can Edit This Automation?","type":"textarea","required":True,"tier_weight":None,
             "placeholder":"Co-owners in Power Platform; Studio users with folder access; OWNERSHIP grantees in Snowflake"},
            {"id":"E3","label":"Principle of Least Privilege Applied?","type":"dropdown","tier_weight":None,
             "options":["","Yes","Partial — gaps documented","No"]},
            {"id":"E4","label":"Power Apps: 'Share with Everyone' Disabled?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a Power Apps automation"]},
            {"id":"E5","label":"Copilot Studio: Sharing Restrictions Configured?","type":"textarea","tier_weight":None,
             "placeholder":"N/A if not applicable — note sharing rules, auth type, max viewer count"},
            {"id":"E6","label":"MFA Enforced for All Interactive Users?","type":"dropdown","tier_weight":None,
             "options":["","Yes","No","N/A — No interactive users"]},
            {"id":"E7","label":"Snowflake Network Policy Applied (user- or account-level)?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a Snowflake automation"]},
        ]
    },
    "F": {
        "title": "Change Management & SDLC", "icon": "🔄",
        "fields": [
            {"id":"F1","label":"Separate Dev / Test / Prod Environments Exist?","tier_weight":"env_separation",
             "type":"dropdown","options":["","Yes","Partial (some separation)","No"]},
            {"id":"F2","label":"Source Code in Version Control (git)?","type":"dropdown","tier_weight":None,
             "options":["","Yes — Azure DevOps","Yes — GitHub","Yes — GitLab","No"]},
            {"id":"F3","label":"Deployment Mechanism","type":"textarea","tier_weight":None,
             "placeholder":"e.g. Power Platform Pipelines, Azure DevOps, UiPath Automation Ops, Snowflake CLI + Git, manual"},
            {"id":"F4","label":"Power Platform: Managed Solutions Used for Test/Prod Deployments?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a Power Platform automation"]},
            {"id":"F5","label":"Documented Rollback Procedure Exists?","type":"dropdown","tier_weight":None,
             "options":["","Yes — tested","Yes — not yet tested","No"]},
            {"id":"F6","label":"UiPath Workflow Analyzer Set to Block Publish on Failing Rules?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a UiPath automation"]},
            {"id":"F7","label":"Snowpark: Unit Tests + Integration Tests in CI Pipeline?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a Snowpark automation"]},
            {"id":"F8","label":"Testing CoE Developed Test Cases (happy-path, edge-case, security abuse)?","tier_weight":None,
             "type":"dropdown","options":["","Yes","In progress","No"]},
        ]
    },
    "G": {
        "title": "Logging, Monitoring & Alerting", "icon": "📊",
        "fields": [
            {"id":"G1","label":"Runtime Errors Logged? Where?","type":"yesno_text","tier_weight":None,
             "hint":"Application Insights, Dataverse, Orchestrator, Snowflake query history"},
            {"id":"G2","label":"Application Insights Enabled for Cloud Flow Telemetry?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a cloud flow"],
             "hint":"Available in Managed Environments"},
            {"id":"G3","label":"Audit Events Ingested into Microsoft Purview?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A"]},
            {"id":"G4","label":"Alerting on Automation Failure Configured?","type":"yesno_text","tier_weight":None,
             "placeholder":"Mechanism: email, Teams, Slack, PagerDuty, etc."},
            {"id":"G5","label":"UiPath: REFramework Pattern with Structured Exception Handling?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a UiPath automation"]},
            {"id":"G6","label":"Snowflake: EXTERNAL_ACCESS_HISTORY Monitored + Query Tagging?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a Snowflake automation"]},
        ]
    },
    "H": {
        "title": "Business Continuity & DR", "icon": "🔁",
        "fields": [
            {"id":"H1","label":"RTO — Recovery Time Objective (hours of downtime tolerated)","type":"text","required":True,
             "placeholder":"e.g. 4 hours, 24 hours, 1 business day","tier_weight":"rto"},
            {"id":"H2","label":"RPO — Recovery Point Objective (maximum acceptable data loss)","type":"text","required":True,
             "placeholder":"e.g. 1 hour, 24 hours, last daily backup","tier_weight":None},
            {"id":"H3","label":"Dataverse: Managed Environments Enabled (28-day retention / self-service DR)?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not Dataverse-backed"]},
            {"id":"H4","label":"Recovery Runbook Documented and Tested?","type":"dropdown","tier_weight":None,
             "options":["","Yes — tested in last 12 months","Yes — not yet tested","No"]},
            {"id":"H5","label":"Snowflake: Time Travel and Fail-Safe Retention Verified?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — Not a Snowflake automation"]},
        ]
    },
    "I": {
        "title": "Compliance & Regulatory", "icon": "⚖️",
        "fields": [
            {"id":"I1","label":"Applicable Regulations (confirm from B8)","type":"textarea","tier_weight":None,
             "placeholder":"List all confirmed applicable regulations and relevant clauses"},
            {"id":"I2","label":"In Scope for a Financial-Reporting Control (SOX ITGC)?","tier_weight":"sox",
             "type":"dropdown","required":True,"options":["","Yes","No"]},
            {"id":"I3","label":"Automation Makes Automated Decisions Affecting Customers / Employees?","tier_weight":"automated_decisions",
             "type":"dropdown","options":["","Yes","No"],
             "hint":"AI fairness / GDPR Article 22 / EU AI Act"},
            {"id":"I4","label":"Data Protection Impact Assessment (DPIA) Required?","type":"dropdown","tier_weight":None,
             "options":["","Yes — completed","Yes — in progress","No"]},
            {"id":"I5","label":"Right to Be Forgotten: Data Deletion Procedure Defined?","type":"yesno_text","tier_weight":None,
             "hint":"Required where personal data is processed"},
            {"id":"I6","label":"Copilot Studio / LLM-Based: Microsoft Purview DSPM-for-AI Policies Applied?","tier_weight":None,
             "type":"dropdown","options":["","Yes","No","N/A — No generative AI components"]},
        ]
    },
}

# ── Tier rules ───────────────────────────────────────────────────
TIER_RULES = {
    "data_type": {
        3: {"PHI", "PCI", "SOX-relevant", "Export-controlled"},
        2: {"PII", "Financial", "Intellectual Property"},
    },
    "sensitivity": {
        3: {"Restricted"},
        2: {"Confidential"},
    },
    "lifespan": {3: {"Mission-critical"}, 2: set()},
    "implicit_sharing": {2: {"Yes"}},
    "egress": {2: {"Yes"}},
    "secrets_in_code": {3: {"Yes"}},
    "env_separation": {2: {"Partial (some separation)", "No"}},
    "sox": {3: {"Yes"}},
    "automated_decisions": {2: {"Yes"}},
}

# ── Control catalog (8 domains, R=required ✅  C=conditional 🟡  O=optional ⚪  -=N/A) ──
CONTROL_CATALOG = [
    {
        "domain": "1. Data Access & Classification",
        "controls": [
            {"id":"D1.1","text":"Data inventory documented & signed by Data Owner","T1":"R","T2":"R","T3":"R","authority":"NIST CSF ID.AM; ISO 27001 A.5.9"},
            {"id":"D1.2","text":"Data classification label applied (Public/Internal/Confidential/Restricted)","T1":"R","T2":"R","T3":"R","authority":"ISO 27001 A.5.12"},
            {"id":"D1.3","text":"Microsoft Purview sensitivity labels on Dataverse/SharePoint outputs","T1":"C","T2":"R","T3":"R","authority":"Microsoft Purview DSPM for AI"},
            {"id":"D1.4","text":"Snowflake tag-based masking on PII/PHI/PCI columns","T1":"C","T2":"R","T3":"R","authority":"Snowflake tag-based masking"},
            {"id":"D1.5","text":"Snowflake row access policies for multi-tenant data filtering","T1":"O","T2":"C","T3":"R","authority":"Snowflake PII guide"},
            {"id":"D1.6","text":"Data residency verified (region matches policy)","T1":"R","T2":"R","T3":"R","authority":"GDPR / Copilot Studio data residency"},
            {"id":"D1.7","text":"Retention & disposition policy defined","T1":"C","T2":"R","T3":"R","authority":"ISO 27001 A.5.33"},
            {"id":"D1.8","text":"Data minimisation: only required fields queried/written","T1":"R","T2":"R","T3":"R","authority":"GDPR Art.5(1)(c)"},
        ]
    },
    {
        "domain": "2. API Connection & Endpoint Security",
        "controls": [
            {"id":"D2.1","text":"All connectors classified under active DLP policy","T1":"R","T2":"R","T3":"R","authority":"Microsoft Learn — DLP strategy"},
            {"id":"D2.2","text":"No implicit connection sharing in Power Apps","T1":"R","T2":"R","T3":"R","authority":"Microsoft — Best security practices for Power Apps"},
            {"id":"D2.3","text":"Endpoint filtering for HTTP/SQL/SMTP/SharePoint connectors","T1":"O","T2":"C","T3":"R","authority":"Microsoft Power Platform Blog — Endpoint filtering"},
            {"id":"D2.4","text":"Custom connector host URL reviewed and classified in DLP","T1":"C","T2":"R","T3":"R","authority":"Microsoft Learn — Custom connector parity"},
            {"id":"D2.5","text":"Snowflake EAI references least-privilege egress rules only","T1":"-","T2":"R","T3":"R","authority":"Snowflake — Creating EAI"},
            {"id":"D2.6","text":"TLS 1.2+ enforced for all external endpoints","T1":"R","T2":"R","T3":"R","authority":"UiPath FedRAMP TLS 1.2"},
            {"id":"D2.7","text":"Cross-tenant restrictions enabled where applicable","T1":"C","T2":"R","T3":"R","authority":"Microsoft Learn — IAM"},
        ]
    },
    {
        "domain": "3. Credential Management",
        "controls": [
            {"id":"D3.1","text":"No secrets in code, config files, or maker-visible env vars","T1":"R","T2":"R","T3":"R","authority":"OWASP LCNC-SEC-04; Snowflake Python best practice"},
            {"id":"D3.2","text":"Secrets stored in enterprise vault (Key Vault / AWS SM / CyberArk / etc.)","T1":"R","T2":"R","T3":"R","authority":"UiPath Credential Stores"},
            {"id":"D3.3","text":"Power Automate env var secrets use Azure Key Vault reference","T1":"R","T2":"R","T3":"R","authority":"Microsoft Learn — Audit logs Azure Key Vault"},
            {"id":"D3.4","text":"Snowflake service accounts use key-pair or OAuth/WIF; TYPE=SERVICE set","T1":"-","T2":"R","T3":"R","authority":"Snowflake Authentication; Keebo"},
            {"id":"D3.5","text":"Key/certificate rotation scheduled (≤180 days T2; ≤90 days T3)","T1":"-","T2":"R","T3":"R","authority":"Snowflake Key-Pair Rotation"},
            {"id":"D3.6","text":"Named owner for each service account + human backup","T1":"R","T2":"R","T3":"R","authority":"ISO 27001 A.5.16"},
            {"id":"D3.7","text":"Deprovisioning procedure defined when maker leaves","T1":"R","T2":"R","T3":"R","authority":"CoE Starter Kit pattern; ISO 27001 A.6.5"},
        ]
    },
    {
        "domain": "4. Change Management / SDLC",
        "controls": [
            {"id":"D4.1","text":"Separate dev, test, and prod environments","T1":"C","T2":"R","T3":"R","authority":"Microsoft Learn — ALM"},
            {"id":"D4.2","text":"Source control (git) for solution/XAML/SQL/Python artifacts","T1":"C","T2":"R","T3":"R","authority":"NIST SSDF PS.1"},
            {"id":"D4.3","text":"Power Platform Managed Solutions deployed to Test/Prod","T1":"-","T2":"R","T3":"R","authority":"Binary Republik"},
            {"id":"D4.4","text":"Managed Environments enabled for Test/Prod (Dataverse-backed)","T1":"-","T2":"R","T3":"R","authority":"Microsoft Learn — Managed Environments"},
            {"id":"D4.5","text":"CI/CD pipeline (Power Platform Pipelines / UiPath Automation Ops / Snowflake CLI)","T1":"C","T2":"R","T3":"R","authority":"UiPath Automation Ops"},
            {"id":"D4.6","text":"UiPath Workflow Analyzer rules passing; Studio policies enforced","T1":"R","T2":"R","T3":"R","authority":"UiPath Docs"},
            {"id":"D4.7","text":"Power CAT Automated Code Review + DocuPilot documentation run","T1":"C","T2":"R","T3":"R","authority":"Power CAT"},
            {"id":"D4.8","text":"Snowpark unit tests + integration tests in CI pipeline","T1":"-","T2":"R","T3":"R","authority":"Infinite Lambda"},
            {"id":"D4.9","text":"Documented rollback procedure","T1":"C","T2":"R","T3":"R","authority":"NIST SSDF PS.3"},
            {"id":"D4.10","text":"Testing CoE sign-off on test plan & results","T1":"C","T2":"R","T3":"R","authority":"Power Platform WAF SE:09"},
            {"id":"D4.11","text":"Vulnerable/outdated components inventoried (NuGet, pip, connectors)","T1":"C","T2":"R","T3":"R","authority":"OWASP LCNC-SEC-07"},
        ]
    },
    {
        "domain": "5. Identity & Access Management",
        "controls": [
            {"id":"D5.1","text":"Access managed via Entra ID security groups (not individual assignments)","T1":"R","T2":"R","T3":"R","authority":"Microsoft Learn — Copilot Studio"},
            {"id":"D5.2","text":"Least-privilege mapping of Dataverse / UiPath folder / Snowflake roles","T1":"R","T2":"R","T3":"R","authority":"Snowflake Access Control"},
            {"id":"D5.3","text":"Snowflake role hierarchy terminates below SYSADMIN; ACCOUNTADMIN not used for runs","T1":"-","T2":"R","T3":"R","authority":"Snowflake Access Control Best Practices"},
            {"id":"D5.4","text":"MFA enforced for interactive users","T1":"R","T2":"R","T3":"R","authority":"Keebo — Snowflake MFA"},
            {"id":"D5.5","text":"PIM / Just-in-time elevation for admin roles","T1":"C","T2":"R","T3":"R","authority":"Microsoft Learn — Secure default environment"},
            {"id":"D5.6","text":"Snowflake network policy (user- or account-level) in place","T1":"-","T2":"R","T3":"R","authority":"Snowflake Network Policies"},
            {"id":"D5.7","text":"Power Platform IP firewall enabled (Managed Environments)","T1":"-","T2":"C","T3":"R","authority":"Microsoft Learn — IAM"},
            {"id":"D5.8","text":"Guest access to Dataverse explicitly approved only where justified","T1":"R","T2":"R","T3":"R","authority":"Microsoft Learn"},
            {"id":"D5.9","text":"Sharing limits configured; 'Share with Everyone' disabled","T1":"R","T2":"R","T3":"R","authority":"Microsoft Learn — Secure default environment"},
            {"id":"D5.10","text":"Quarterly access reviews for automation owners/editors","T1":"-","T2":"R","T3":"R","authority":"ISO 27001 A.5.18"},
        ]
    },
    {
        "domain": "6. Logging, Monitoring & Alerting",
        "controls": [
            {"id":"D6.1","text":"Audit logging enabled at environment level (Dataverse auditing ON)","T1":"R","T2":"R","T3":"R","authority":"Microsoft Learn — Dataverse auditing"},
            {"id":"D6.2","text":"Power Automate / Copilot Studio events ingested into Microsoft Purview","T1":"C","T2":"R","T3":"R","authority":"Microsoft Learn — Purview activity logs"},
            {"id":"D6.3","text":"CoE Starter Kit audit-log sync via Office 365 Management API","T1":"C","T2":"R","T3":"R","authority":"Microsoft Learn — CoE audit-log setup"},
            {"id":"D6.4","text":"Application Insights attached to cloud flows (Managed Environments)","T1":"-","T2":"R","T3":"R","authority":"Microsoft Learn"},
            {"id":"D6.5","text":"UiPath Orchestrator logs forwarded to SIEM","T1":"-","T2":"R","T3":"R","authority":"ISO 27001 A.8.15"},
            {"id":"D6.6","text":"Snowflake query tagging + EXTERNAL_ACCESS_HISTORY monitored","T1":"-","T2":"R","T3":"R","authority":"Snowflake Docs"},
            {"id":"D6.7","text":"Alerts on failure (duration-based, error-count-based)","T1":"R","T2":"R","T3":"R","authority":"NIST CSF DE.CM"},
            {"id":"D6.8","text":"UiPath REFramework error-screenshot capture + exception routing","T1":"C","T2":"R","T3":"R","authority":"UiPath REFramework"},
            {"id":"D6.9","text":"Sensitive data (PII/credentials) masked in logs","T1":"R","T2":"R","T3":"R","authority":"OWASP LCNC-SEC-03"},
            {"id":"D6.10","text":"Log retention aligned to policy (90-day Purview default extended as required)","T1":"R","T2":"R","T3":"R","authority":"Microsoft Learn"},
        ]
    },
    {
        "domain": "7. Business Continuity & Disaster Recovery",
        "controls": [
            {"id":"D7.1","text":"Documented RTO and RPO","T1":"C","T2":"R","T3":"R","authority":"ISO 22301; Microsoft Learn — BCDR"},
            {"id":"D7.2","text":"Manual backup taken before major releases (Dataverse)","T1":"C","T2":"R","T3":"R","authority":"Microsoft Learn — Back up and restore environments"},
            {"id":"D7.3","text":"Self-service DR enabled for Dataverse-backed production environments","T1":"-","T2":"C","T3":"R","authority":"Microsoft Learn — BCDR"},
            {"id":"D7.4","text":"Tested rollback/restore in the last 12 months","T1":"-","T2":"R","T3":"R","authority":"NIST CSF RC.RP"},
            {"id":"D7.5","text":"Snowflake Time Travel / Fail-safe retention tested","T1":"-","T2":"C","T3":"R","authority":"Snowflake Securing Snowflake overview"},
            {"id":"D7.6","text":"Runbook published and owner identified","T1":"C","T2":"R","T3":"R","authority":"ISO 27001 A.5.29"},
            {"id":"D7.7","text":"BCDR tabletop exercise at least annually","T1":"-","T2":"-","T3":"R","authority":"NIST CSF RC.IM"},
        ]
    },
    {
        "domain": "8. Compliance & Regulatory",
        "controls": [
            {"id":"D8.1","text":"Applicable regulations listed and mapped to controls","T1":"R","T2":"R","T3":"R","authority":"NIST CSF GV.OC"},
            {"id":"D8.2","text":"DPIA completed where personal data is in scope","T1":"-","T2":"C","T3":"R","authority":"GDPR Art. 35"},
            {"id":"D8.3","text":"SOX ITGC mapping (change mgmt, access, ops)","T1":"-","T2":"C","T3":"R","authority":"PCAOB AS 2201"},
            {"id":"D8.4","text":"Copilot Studio / generative-AI: Purview DSPM-for-AI policies applied","T1":"-","T2":"R","T3":"R","authority":"Microsoft Learn — Copilot Control System"},
            {"id":"D8.5","text":"Agentic AI workloads mapped against OWASP Top 10 for Agentic Applications","T1":"-","T2":"R","T3":"R","authority":"Microsoft Security Blog"},
            {"id":"D8.6","text":"Records-retention alignment (legal hold capability preserved)","T1":"C","T2":"R","T3":"R","authority":"ISO 27001 A.5.33"},
            {"id":"D8.7","text":"Export-control / data-sovereignty confirmed for cross-border flows","T1":"C","T2":"R","T3":"R","authority":"UiPath FedRAMP"},
            {"id":"D8.8","text":"Exception register maintained for unmet controls (justification + compensating control + expiry)","T1":"R","T2":"R","T3":"R","authority":"Hyperproof"},
        ]
    },
]

In [ ]:

# ── Widget builder ────────────────────────────────────────────────
def build_widget_for_field(field: dict) -> widgets.Widget:
    fid = field["id"]
    ftype = field["type"]
    req_star = "<span style='color:red'> *</span>" if field.get("required") else ""
    label_html = f"<b>{field['label']}</b>{req_star}"
    hint_html = (f"<small style='color:#6c757d'>{field['hint']}</small>"
                 if field.get("hint") else "")

    if ftype == "text":
        w = widgets.Text(placeholder=field.get("placeholder", ""),
                         layout=widgets.Layout(width="520px"))
    elif ftype == "textarea":
        w = widgets.Textarea(placeholder=field.get("placeholder", ""),
                             layout=widgets.Layout(width="520px", height="80px"))
    elif ftype == "dropdown":
        w = widgets.Dropdown(options=field["options"],
                             layout=widgets.Layout(width="360px"))
    elif ftype == "multicheck":
        cbs = [widgets.Checkbox(description=opt, indent=False, value=False,
                                layout=widgets.Layout(width="250px"))
               for opt in field["options"]]
        w = widgets.GridBox(cbs,
                            layout=widgets.Layout(
                                grid_template_columns="repeat(auto-fill, 250px)",
                                gap="2px"))
        w._cbs = cbs
        w._opts = field["options"]
    elif ftype in ("yesno", "yesno_text"):
        toggle = widgets.ToggleButtons(options=["Yes", "No", "N/A"],
                                       button_style="",
                                       layout=widgets.Layout(width="auto"))
        notes = widgets.Textarea(placeholder=field.get("placeholder", "Notes / details…"),
                                 layout=widgets.Layout(
                                     width="520px", height="56px", display="none"))
        def _on_toggle(change, _n=notes):
            _n.layout.display = "flex" if change["new"] == "Yes" else "none"
        toggle.observe(_on_toggle, names="value")
        w = widgets.VBox([toggle, notes])
        w._toggle = toggle
        w._notes = notes
    elif ftype == "date":
        w = widgets.DatePicker(layout=widgets.Layout(width="200px"))
    else:
        w = widgets.Text(layout=widgets.Layout(width="520px"))

    _WIDGET_REGISTRY[fid] = w
    rows = [widgets.HTML(label_html), w]
    if hint_html:
        rows.append(widgets.HTML(hint_html))
    return widgets.VBox(rows, layout=widgets.Layout(margin="0 0 14px 0"))


def build_section_widget(section_key: str) -> widgets.VBox:
    s = QUESTIONNAIRE_SCHEMA[section_key]
    header = widgets.HTML(
        f"<h3 style='margin:0 0 14px 0;color:#0c3483'>"
        f"{s['icon']} Section {section_key}: {s['title']}</h3>")
    children = [header] + [build_widget_for_field(f) for f in s["fields"]]
    return widgets.VBox(children, layout=widgets.Layout(padding="16px"))


# ── Answer collector ──────────────────────────────────────────────
def collect_answers() -> dict:
    out = {}
    for fid, w in _WIDGET_REGISTRY.items():
        if isinstance(w, widgets.GridBox) and hasattr(w, "_cbs"):
            out[fid] = [cb.description for cb in w._cbs if cb.value]
        elif isinstance(w, widgets.VBox) and hasattr(w, "_toggle"):
            out[fid] = w._toggle.value
            out[f"{fid}_notes"] = w._notes.value
        elif isinstance(w, widgets.DatePicker):
            out[fid] = str(w.value) if w.value else ""
        else:
            out[fid] = w.value if hasattr(w, "value") else ""
    return out


# ── Tier calculator ───────────────────────────────────────────────
def calculate_tier(answers: dict) -> tuple:
    tier, reasons = 1, []

    def promote(t: int, reason: str):
        nonlocal tier
        if t > tier:
            tier = t
        reasons.append(f"→ Tier {t}: {reason}")

    # Data type (B3)
    dt = set(answers.get("B3", []))
    if dt & TIER_RULES["data_type"][3]:
        promote(3, f"Regulated data: {', '.join(dt & TIER_RULES['data_type'][3])}")
    elif dt & TIER_RULES["data_type"][2]:
        promote(2, f"Sensitive data: {', '.join(dt & TIER_RULES['data_type'][2])}")

    # Sensitivity (B4)
    sens = answers.get("B4", "")
    if sens in TIER_RULES["sensitivity"][3]:
        promote(3, f"Sensitivity: {sens}")
    elif sens in TIER_RULES["sensitivity"].get(2, set()):
        promote(2, f"Sensitivity: {sens}")

    # Compliance frameworks (B8)
    regs = [r for r in answers.get("B8", []) if r != "None"]
    if len(regs) >= 2:
        promote(3, f"Multiple regulatory frameworks: {', '.join(regs)}")
    elif len(regs) == 1:
        promote(2, f"Regulatory framework in scope: {regs[0]}")

    # Mission-critical lifespan (A8)
    if answers.get("A8", "") in TIER_RULES["lifespan"][3]:
        promote(3, "Mission-critical lifespan (A8)")

    # Implicit sharing (C3)
    if answers.get("C3", "") == "Yes":
        promote(2, "Implicit connection sharing detected (C3)")

    # External egress (C4)
    if answers.get("C4", "") == "Yes":
        promote(2, "External egress calls outside corporate tenant (C4)")

    # Secrets in code (D1) — critical
    if answers.get("D1", "") == "Yes":
        promote(3, "Secrets stored in code / config — critical finding (D1)")

    # Insufficient env separation (F1)
    if answers.get("F1", "") in TIER_RULES["env_separation"][2]:
        promote(2, f"Insufficient environment separation: {answers.get('F1', '')} (F1)")

    # SOX ITGC (I2)
    if answers.get("I2", "") == "Yes":
        promote(3, "SOX ITGC in scope (I2)")

    # Automated decisions (I3)
    if answers.get("I3", "") == "Yes":
        promote(2, "Automated decisions affecting customers/employees (I3)")

    _LAST_TIER[0] = tier
    return tier, reasons


# ── Excel template generator ──────────────────────────────────────
def generate_excel_template() -> bytes:
    wb = Workbook()
    ws = wb.active
    ws.title = "MSR_Questionnaire"

    hdr_fill = PatternFill("solid", fgColor="0C3483")
    hdr_font = Font(color="FFFFFF", bold=True)
    for col, h in enumerate(["field_id", "section", "question", "answer", "notes"], 1):
        c = ws.cell(row=1, column=col, value=h)
        c.fill = hdr_fill
        c.font = hdr_font
        c.alignment = Alignment(horizontal="center")

    row = 2
    for sk, sv in QUESTIONNAIRE_SCHEMA.items():
        for field in sv["fields"]:
            ws.cell(row=row, column=1, value=field["id"])
            ws.cell(row=row, column=2, value=f"{sk}: {sv['title']}")
            ws.cell(row=row, column=3, value=field["label"])
            ws.cell(row=row, column=4, value="")
            ftype = field["type"]
            if ftype == "multicheck":
                note = f"Multi-select, comma-separate. Options: {', '.join(field['options'])}"
            elif ftype == "dropdown":
                note = f"Single select. Options: {', '.join(str(o) for o in field['options'] if o)}"
            elif ftype in ("yesno", "yesno_text"):
                note = "Enter: Yes / No / N/A"
            elif ftype == "date":
                note = "Format: YYYY-MM-DD"
            else:
                note = field.get("hint", "")
            ws.cell(row=row, column=5, value=note)
            row += 1

    ws.column_dimensions["A"].width = 10
    ws.column_dimensions["B"].width = 30
    ws.column_dimensions["C"].width = 55
    ws.column_dimensions["D"].width = 35
    ws.column_dimensions["E"].width = 55
    ws.freeze_panes = "A2"

    buf = __import__("io").BytesIO()
    wb.save(buf)
    return buf.getvalue()


# ── Excel upload parser ───────────────────────────────────────────
def parse_excel_upload(content: bytes) -> dict:
    df = pd.read_excel(__import__("io").BytesIO(content),
                       sheet_name="MSR_Questionnaire", dtype=str).fillna("")
    return {str(r.get("field_id", "")).strip(): str(r.get("answer", "")).strip()
            for _, r in df.iterrows() if r.get("field_id", "").strip()}


# ── Widget populator ──────────────────────────────────────────────
def populate_widgets_from_dict(answer_dict: dict):
    for fid, raw in answer_dict.items():
        if fid not in _WIDGET_REGISTRY or not raw:
            continue
        w = _WIDGET_REGISTRY[fid]
        if isinstance(w, widgets.GridBox) and hasattr(w, "_cbs"):
            selected = {v.strip() for v in raw.split(",")}
            for cb in w._cbs:
                cb.value = cb.description in selected
        elif isinstance(w, widgets.VBox) and hasattr(w, "_toggle"):
            if raw in ("Yes", "No", "N/A"):
                w._toggle.value = raw
        elif isinstance(w, widgets.DatePicker):
            try:
                w.value = datetime.strptime(raw, "%Y-%m-%d").date()
            except Exception:
                pass
        else:
            try:
                w.value = raw
            except Exception:
                pass


# ── HTML report generator ─────────────────────────────────────────
def generate_html_report(answers: dict, tier: int, reasons: list) -> str:
    tc = {1: "#28a745", 2: "#fd7e14", 3: "#dc3545"}
    tl = {1: "Tier 1 — Low Risk", 2: "Tier 2 — Medium Risk", 3: "Tier 3 — High / Mission-critical"}
    tap = {
        1: ["CoE Lead"],
        2: ["CoE Lead", "Data Owner"],
        3: ["Business Sponsor", "Data Owner", "InfoSec", "Privacy / Legal", "CoE Lead", "Testing CoE"],
    }
    tr_desc = {
        1: "Self-attestation + CoE lightweight review. Maker self-attestation ✔, CoE review recommended.",
        2: "CoE technical review required + Power CAT / Workflow Analyzer / pytest gates + Data-Owner sign-off.",
        3: "Full Mini-MSR engagement: InfoSec + Privacy + Compliance + Business Sponsor + Threat model (OWASP LCNC + Agentic Top-10) + Pen/abuse test + BCDR tabletop.",
    }
    ci = {"R": "✅", "C": "🟡", "O": "⚪", "-": "—"}
    tk = {1: "T1", 2: "T2", 3: "T3"}[tier]

    auto_name = answers.get("A1", "Unnamed Automation") or "Unnamed Automation"
    today = date.today().strftime("%Y-%m-%d")

    # Build control checklist
    ctl_html = ""
    for dom in CONTROL_CATALOG:
        rows = ""
        for c in dom["controls"]:
            st = c[tk]
            icon = ci.get(st, "—")
            bg = "background:#fff3cd" if st == "C" else ("background:#fde8e8" if st == "R" else "")
            rows += (f"<tr style='{bg}'>"
                     f"<td style='padding:5px 8px;font-size:12px;color:#555;width:56px'>{c['id']}</td>"
                     f"<td style='padding:5px 8px;font-size:13px'>{c['text']}</td>"
                     f"<td style='padding:5px 8px;text-align:center;font-size:15px;width:52px'>{icon}</td>"
                     f"<td style='padding:5px 8px;font-size:11px;color:#6c757d;width:210px'>{c['authority']}</td>"
                     f"</tr>")
        ctl_html += (f"<h4 style='margin:18px 0 6px;color:#0c3483'>{dom['domain']}</h4>"
                     f"<table style='width:100%;border-collapse:collapse;border:1px solid #dee2e6;margin-bottom:6px'>"
                     f"<tr style='background:#0c3483;color:white'>"
                     f"<th style='padding:6px'>ID</th><th style='padding:6px'>Control</th>"
                     f"<th style='padding:6px'>Status</th><th style='padding:6px'>Authority</th></tr>"
                     f"{rows}</table>")

    # Build questionnaire summary
    qs_html = ""
    for sk, sv in QUESTIONNAIRE_SCHEMA.items():
        rows = ""
        for field in sv["fields"]:
            val = answers.get(field["id"], "")
            if isinstance(val, list):
                val = ", ".join(val) if val else "—"
            val = val or "—"
            rows += (f"<tr><td style='padding:6px;font-weight:500;width:38%;background:#f8f9fa;"
                     f"vertical-align:top'>{field['id']}: {field['label']}</td>"
                     f"<td style='padding:6px;vertical-align:top'>{val}</td></tr>")
        qs_html += (f"<details style='margin-bottom:6px'>"
                    f"<summary style='cursor:pointer;padding:8px;background:#e9ecef;"
                    f"border-radius:4px;font-weight:bold'>"
                    f"{sv['icon']} Section {sk}: {sv['title']}</summary>"
                    f"<table style='width:100%;border-collapse:collapse;"
                    f"border:1px solid #dee2e6'>{rows}</table></details>")

    # Approvals
    ap_html = "".join(
        f"<tr><td style='padding:12px 16px;border-right:1px solid #dee2e6'><b>{role}</b></td>"
        f"<td style='padding:12px 16px;width:200px'>________________</td>"
        f"<td style='padding:12px 16px;width:120px'>________________</td></tr>"
        for role in tap[tier]
    )

    # Reason list
    reasons_li = "".join(f"<li>{r}</li>" for r in reasons) if reasons else         "<li>No elevated risk factors identified — defaulting to Tier 1.</li>"

    platforms = ", ".join(answers.get("A4", []) or ["—"])
    biz_owner = answers.get("A2", "—") or "—"
    tech_lead = answers.get("A3", "—") or "—"
    sensitivity = answers.get("B4", "—") or "—"
    go_live = answers.get("A7", "—") or "—"

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Mini-MSR — {auto_name}</title>
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
          margin: 0; padding: 0; color: #212529; background: #fff; }}
  .page {{ max-width: 960px; margin: 0 auto; padding: 24px; }}
  h2 {{ color: #0c3483; border-bottom: 2px solid #0c3483; padding-bottom: 6px; margin-top: 28px; }}
  table {{ border-spacing: 0; }}
  @media print {{
    .no-print {{ display: none; }}
    details {{ display: block; }}
    details summary {{ list-style: none; }}
  }}
</style>
</head>
<body>
<div class="page">

<div style="background:linear-gradient(135deg,#0c3483 0%,#a2b6df 100%);
            padding:24px;border-radius:8px;color:white;margin-bottom:24px;
            display:flex;justify-content:space-between;align-items:flex-start;flex-wrap:wrap;gap:12px">
  <div>
    <h1 style="margin:0 0 4px">🔒 Mini-MSR Security Review Report</h1>
    <h2 style="margin:0;font-size:18px;opacity:0.9;border:none;color:white">{auto_name}</h2>
  </div>
  <div style="text-align:right">
    <div style="background:{tc[tier]};color:white;padding:8px 20px;border-radius:20px;
                font-size:20px;font-weight:bold;white-space:nowrap">{tl[tier]}</div>
    <div style="opacity:0.85;margin-top:6px;font-size:13px">Generated: {today}</div>
  </div>
</div>

<table style="width:100%;border-collapse:collapse;margin-bottom:24px;border:1px solid #dee2e6">
  <tr style="background:#f8f9fa">
    <th style="padding:10px;text-align:left;border-right:1px solid #dee2e6;width:30%">Field</th>
    <th style="padding:10px;text-align:left">Value</th>
  </tr>
  <tr><td style="padding:10px;border-right:1px solid #dee2e6;font-weight:500">Business Owner</td>
      <td style="padding:10px">{biz_owner}</td></tr>
  <tr style="background:#f8f9fa">
    <td style="padding:10px;border-right:1px solid #dee2e6;font-weight:500">Technical Lead</td>
    <td style="padding:10px">{tech_lead}</td></tr>
  <tr><td style="padding:10px;border-right:1px solid #dee2e6;font-weight:500">Platforms</td>
      <td style="padding:10px">{platforms}</td></tr>
  <tr style="background:#f8f9fa">
    <td style="padding:10px;border-right:1px solid #dee2e6;font-weight:500">Data Sensitivity</td>
    <td style="padding:10px">{sensitivity}</td></tr>
  <tr><td style="padding:10px;border-right:1px solid #dee2e6;font-weight:500">Target Go-Live</td>
      <td style="padding:10px">{go_live}</td></tr>
</table>

<h2>📊 Tier Determination</h2>
<div style="background:#fff8e1;border:2px solid {tc[tier]};border-radius:8px;padding:20px;margin-bottom:8px">
  <div style="font-size:22px;font-weight:bold;color:{tc[tier]};margin-bottom:10px">{tl[tier]}</div>
  <p style="margin:0 0 8px"><b>Scoring rationale:</b></p>
  <ul style="margin:0 0 12px;padding-left:20px">{reasons_li}</ul>
  <p style="margin:0;font-size:13px;color:#555"><b>Required review intensity:</b> {tr_desc[tier]}</p>
</div>

<h2>✅ Control Checklist — {tl[tier]}</h2>
<p style="margin:0 0 12px;font-size:13px">
  ✅ Required &nbsp;&nbsp; 🟡 Conditional &nbsp;&nbsp; ⚪ Optional &nbsp;&nbsp; — Not applicable at this tier
</p>
{ctl_html}

<h2>📋 Questionnaire Summary</h2>
{qs_html}

<h2>⚠️ Exception Register</h2>
<p style="font-size:13px;color:#555">
  Log any Required (✅) controls that cannot be met. Each exception requires: business justification,
  compensating control, expiry date, and named accepting executive.
</p>
<table style="width:100%;border-collapse:collapse;border:1px solid #dee2e6">
  <tr style="background:#0c3483;color:white">
    <th style="padding:8px">Control ID</th>
    <th style="padding:8px">Description</th>
    <th style="padding:8px">Business Justification</th>
    <th style="padding:8px">Compensating Control</th>
    <th style="padding:8px">Expiry Date</th>
    <th style="padding:8px">Accepting Executive</th>
  </tr>
  <tr><td style="padding:24px;color:#aaa;text-align:center" colspan="6">
    <i>Add rows for each unmet Required control</i>
  </td></tr>
</table>

<h2>✍️ Approval Block</h2>
<table style="width:100%;border-collapse:collapse;border:1px solid #dee2e6">
  <tr style="background:#0c3483;color:white">
    <th style="padding:10px;text-align:left">Approver Role</th>
    <th style="padding:10px;text-align:left">Signature</th>
    <th style="padding:10px;text-align:left">Date</th>
  </tr>
  {ap_html}
</table>

<p style="margin-top:32px;font-size:11px;color:#aaa;text-align:center">
  Generated by Mini-MSR Notebook · {today} ·
  NIST CSF 2.0 · ISO/IEC 27001:2022 · OWASP LCNC Top 10
</p>

</div>
</body>
</html>"""


In [ ]:

# ── Excel import tab ─────────────────────────────────────────────
_upload = widgets.FileUpload(accept=".xlsx", multiple=False,
                              layout=widgets.Layout(width="auto"))
_upload_status = widgets.HTML("")
_tmpl_btn = widgets.Button(description="⬇️ Download Blank Template",
                            button_style="info", icon="download",
                            layout=widgets.Layout(height="34px"))
_tmpl_out = widgets.Output()

def _on_tmpl(b):
    b64 = __import__("base64").b64encode(generate_excel_template()).decode()
    with _tmpl_out:
        clear_output(wait=True)
        display(HTML(
            "<a href='data:application/vnd.openxmlformats-officedocument"
            ".spreadsheetml.sheet;base64," + b64 + "' "
            "download='mini_msr_template.xlsx' "
            "style='display:inline-block;padding:6px 14px;background:#17a2b8;"
            "color:white;text-decoration:none;border-radius:4px'>"
            "📥 mini_msr_template.xlsx</a>"))

_tmpl_btn.on_click(_on_tmpl)

def _on_upload(change):
    if not _upload.value:
        return
    content = list(_upload.value.values())[0]["content"]
    try:
        d = parse_excel_upload(bytes(content))
        populate_widgets_from_dict(d)
        _upload_status.value = (f"<span style='color:#28a745'>✅ Imported "
                                f"{len(d)} fields successfully</span>")
    except Exception as e:
        _upload_status.value = f"<span style='color:#dc3545'>❌ Import error: {e}</span>"

_upload.observe(_on_upload, names="value")

_import_tab = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 12px'>📂 Excel Import</h3>"
                 "<p><b>Step 1:</b> Download the blank template, fill it in Excel, "
                 "then upload it to auto-populate all form fields.</p>"),
    widgets.HBox([_tmpl_btn, _tmpl_out],
                 layout=widgets.Layout(gap="10px", align_items="center")),
    _DIVIDER,
    widgets.HTML("<p><b>Step 2:</b> Upload your completed file (.xlsx):</p>"),
    _upload,
    _upload_status,
], layout=widgets.Layout(padding="16px"))

# ── Build all section tabs ─────────────────────────────────────────
_tab_children = []
_tab_titles = []
for _sk, _sv in QUESTIONNAIRE_SCHEMA.items():
    _tab_children.append(build_section_widget(_sk))
    _tab_titles.append(f"{_sv['icon']} {_sk}")
_tab_children.append(_import_tab)
_tab_titles.append("📂 Import")

_tabs = widgets.Tab(children=_tab_children)
for _i, _t in enumerate(_tab_titles):
    _tabs.set_title(_i, _t)

# ── Action bar ────────────────────────────────────────────────────
_tier_badge = widgets.HTML(
    "<span style='color:#888;padding:4px 10px;background:#f8f9fa;"
    "border-radius:12px;border:1px solid #dee2e6'>Tier not yet calculated</span>")
_gen_btn = widgets.Button(description="📄 Generate Report",
                           button_style="primary",
                           layout=widgets.Layout(height="36px"))
_exp_btn = widgets.Button(description="⬇️ Export HTML",
                           button_style="success", disabled=True,
                           layout=widgets.Layout(height="36px"))
_report_out = widgets.Output()
_export_out = widgets.Output()
_report_cache = {"html": ""}

def _on_generate(b):
    _gen_btn.disabled = True
    _gen_btn.description = "⏳ Generating…"
    try:
        ans = collect_answers()
        t, rs = calculate_tier(ans)
        tc = {1: "#28a745", 2: "#fd7e14", 3: "#dc3545"}
        tl = {1: "Tier 1 — Low Risk", 2: "Tier 2 — Medium Risk", 3: "Tier 3 — High Risk"}
        _tier_badge.value = (f"<span style='background:{tc[t]};color:white;"
                             f"padding:4px 14px;border-radius:12px;font-weight:bold'>"
                             f"{tl[t]}</span>")
        html = generate_html_report(ans, t, rs)
        _report_cache["html"] = html
        _exp_btn.disabled = False
        with _report_out:
            clear_output(wait=True)
            display(HTML(html))
    finally:
        _gen_btn.disabled = False
        _gen_btn.description = "📄 Generate Report"

def _on_export(b):
    b64 = __import__("base64").b64encode(
        _report_cache["html"].encode("utf-8")).decode()
    fname_raw = _WIDGET_REGISTRY.get("A1")
    fname_val = (fname_raw.value if fname_raw and hasattr(fname_raw, "value") else "report")
    fname = "mini_msr_" + fname_val.replace(" ", "_")[:40] + ".html"
    with _export_out:
        clear_output(wait=True)
        display(HTML(
            f"<a href='data:text/html;charset=utf-8;base64,{b64}' "
            f"download='{fname}' "
            "style='display:inline-block;padding:6px 14px;background:#28a745;"
            "color:white;text-decoration:none;border-radius:4px'>"
            f"📥 Download {fname}</a>"))

_gen_btn.on_click(_on_generate)
_exp_btn.on_click(_on_export)

# ── Final display ─────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HTML(
        f"<div style='{_BANNER_STYLE}'>"
        "<h2 style='margin:0 0 4px'>🔒 Mini-MSR Questionnaire</h2>"
        "<p style='margin:0;opacity:0.9'>"
        "Power Platform · UiPath · Snowflake Python Automations</p>"
        "</div>"),
    _tabs,
    _DIVIDER,
    widgets.HBox([_gen_btn, _tier_badge, _exp_btn],
                 layout=widgets.Layout(
                     gap="12px", align_items="center", padding="6px 0")),
    _export_out,
    _report_out,
], layout=widgets.Layout(padding="0")))
